#  Importing Libraries

In [411]:
import os
import pandas as pd
import re

# Load and setup CSV

In [6]:
df = pd.read_csv("observations-704882.csv")
df

,id,url,species_guess,scientific_name,common_name,iconic_taxon_name,taxon_id,"field:url for ""partner"" observation"
0,10686,http://www.inaturalist.org/observations/10686,House Mouse,Mus musculus,House Mouse,Mammalia,44705.0,https://www.inaturalist.org/observations/1766953
1,338604,http://conabio.inaturalist.org/observations/33...,Correcaminos norteño,Geococcyx californianus,Greater Roadrunner,Aves,1986.0,NaN
2,414645,http://www.inaturalist.org/observations/414645,Cape Wolf Snake,Lycophidion capense,Cape Wolf Snake,Reptilia,29508.0,https://www.inaturalist.org/observations/414707
3,414707,http://www.inaturalist.org/observations/414707,Sundevall's Writhing Skink,Mochlus sundevallii,Sundevall's Writhing Skink,Reptilia,37932.0,https://www.inaturalist.org/observations/414645
4,420386,http://www.inaturalist.org/observations/420386,Eastern Kingsnake,Lampropeltis getula,Eastern Kingsnake,Reptilia,29813.0,NaN
...,...,...,...,...,...,...,...,...
14894,347744271,https://www.inaturalist.org/observations/34774...,Ice-Cream-Bean · Guama,Inga edulis,Ice-cream-bean,Plantae,209926.0,https://www.inaturalist.org/observations/34774...
14895,347744274,https://www.inaturalist.org/observations/34774...,Fungi Including Lichens · Hongos,Fungi,Fungi Including Lichens,Fungi,47170.0,https://www.inaturalist.org/observations/34774...
14896,347744290,https://www.inaturalist.org/observations/34774...,Spreading False Pimpernel,Vandellia diffusa,Spreading False Pimpernel,Plantae,1245374.0,https://www.inaturalist.org/observations/34774...
14897,347744291,https://www.inaturalist.org/observations/34774...,Ants · Hormigas,Formicidae,Ants,Insecta,47336.0,https://www.inaturalist.org/observations/34774...


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14899 entries, 0 to 14898
Data columns (total 8 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   id                                   14899 non-null  int64  
 1   url                                  14899 non-null  str    
 2   species_guess                        14591 non-null  str    
 3   scientific_name                      14898 non-null  str    
 4   common_name                          12833 non-null  str    
 5   iconic_taxon_name                    14884 non-null  str    
 6   taxon_id                             14898 non-null  float64
 7   field:url for "partner" observation  12099 non-null  str    
dtypes: float64(1), int64(1), str(6)
memory usage: 931.3 KB


# Get Partner_IDs for each observation

In [346]:
col = 'field:url for "partner" observation'


## Read from csv again

In [462]:
correct = pd.read_csv("correct.csv")
copy = correct.copy()

In [463]:
# correct = copy
# correct.to_csv("correct.csv",index=False)

## Find Invalid URLs

In [415]:
df_temp = df.copy()

In [ ]:
# # 1) Pattern: allow http/https, optional www, any iNaturalist TLD, digits-only ID; ignore trailing ; or , content
pattern = (
    r'(?i)^\s*https?://(?:www\.)?'
    r'(?:(?:[a-z0-9-]+\.)?inaturalist\.org'      # subdomains of inaturalist.org (e.g., uk.inaturalist.org)
    r'|inaturalist\.[a-z]+(?:\.[a-z]+){0,2}'     # inaturalist.<tld> with up to two extra labels (e.g., ala.org.au)
    r'|naturalista\.mx'                           # naturalista.mx
    r'|biodiversity4all\.org)'                    # biodiversity4all.org
    r'/observations/\d+'                          # observations/<digits>
    r'(?:\s*(?:[;,]|\s).*)?\s*$'                  # allow space/comma/semicolon + anything after
)



# 2) Work with a Series; strip spaces to make matching resilient
urls = df_temp[col].astype(str).str.strip()

# 3) Build mask of rows that DO NOT match the required observation URL format
invalid_mask = ~urls.str.match(pattern)

# 4) Get the invalid URLs as a Series
invalid_urls = urls[invalid_mask]

#5) Remove "nan" values
invalid_urls = invalid_urls[invalid_urls.str.lower() != 'nan']

In [351]:
invalid_urls[~(invalid_urls.isnull())]

470      https://www.inaturalist.org/observations/43116...
501      https://www.inaturalist.org/observations/14425...
839                                                     na
848                                                     na
925                                      Linckia laevigata
                               ...                        
14097    https://www.inaturalist.org/observations?on=20...
14334       https://inaturalist.lu//observations/338544252
14335          https://inaturalist.lu//observations/338544
14469             don't have one bc the tree is cultivated
14869               inaturalist.org/observations/347352736
Name: field:url for "partner" observation, Length: 117, dtype: str

In [352]:
invalid_urls.to_csv("invalid_urls.csv", index=False)

In [353]:
len(invalid_urls)
# type(invalid_urls)

2917

## Starting with http

In [354]:
http_df = invalid_urls[invalid_urls.str.startswith('http',na=False)]
len(http_df )


58

## starting with "https://www.inaturalist"

In [355]:
inat = http_df[http_df.str.startswith('https://www.inaturalist',na=False)]

In [356]:
len(inat)

56

In [357]:
inat 

470      https://www.inaturalist.org/observations/43116...
501      https://www.inaturalist.org/observations/14425...
982      https://www.inaturalist.org/observations/new?c...
3710     https://www.inaturalist.org/observations/11339...
4848     https://www.inaturalist.org/observations/14778...
4961     https://www.inaturalist.org/taxa/91566-Pterygo...
5290     https://www.inaturalist.org/taxa/53511-Monopte...
5869     https://www.inaturalist.org/observations/17883...
6084     https://www.inaturalist.org/taxa/85876-Loricar...
6109     https://www.inaturalist.org/observations/18492...
6364     https://www.inaturalist.org/observations/14624...
6365     https://www.inaturalist.org/observations/18717...
6535        https://www.inaturalist.org/guide_taxa/1201268
6744     https://www.inaturalist.org/taxa/238252-Python...
6757     https://www.inaturalist.org/taxa/35342-Iguana-...
6760     https://www.inaturalist.org/taxa/41663-Procyon...
6764               https://www.inaturalist.org/taxa/2847

## Contains observations

In [358]:
inat_obs = inat[inat.str.contains("observations",na=False)]
inat_obs.to_list()

['https://www.inaturalist.org/observations/4311650#activity_comment_2442449',
 'https://www.inaturalist.org/observations/144250537#activity_comment_6ca468b2-ec39-409e-bc11-3af829ae81fa',
 'https://www.inaturalist.org/observations/new?copy=51188925',
 'https://www.inaturalist.org/observations/113398823: https://www.inaturalist.org/observations/113401806; https://www.inaturalist.org/observations/113730771; https://www.inaturalist.org/observations/209436116',
 'https://www.inaturalist.org/observations/147784984#activity_identification_1fe273e8-10d6-40a4-a523-89b2935a285f',
 'https://www.inaturalist.org/observations/178838523#activity_identification_01047684-05e7-488b-b3da-8df97cc82f7f',
 'https://www.inaturalist.org/observations/184926584#activity_identification_f240e66e-4a87-45be-b191-e6b70849da8f',
 'https://www.inaturalist.org/observations/146244000#activity_comment_dee7febf-c164-40b2-9fac-747fdadab26b',
 'https://www.inaturalist.org/observations/187171161#activity_comment_10db2a61-e47

In [359]:
len(inat_obs)

17

In [360]:
type(inat_obs)

pandas.Series

In [361]:

new_rows = pd.DataFrame({
    "Invalid_link": inat_obs,
    "category": "obs",
    "correct": pd.NA
})


In [362]:
correct = pd.concat([correct, new_rows], ignore_index=True).to_csv('correct.csv',index=False)
# .to_csv("correct.csv", index=False)

PermissionError: [Errno 13] Permission denied: 'correct.csv'

In [ ]:
correct = pd.read_csv("correct.csv")
copy = correct.copy()

## Contains "Taxa"

In [ ]:
inat_taxa = inat[inat.str.contains("taxa", na=False)]
len(inat_taxa)

37

In [ ]:


new_rows = pd.DataFrame({
    "Invalid_link": inat_taxa,
    "category": "taxa",
    "correct": pd.NA
})



In [ ]:
correct = pd.concat([correct, new_rows], ignore_index=True).to_csv('correct.csv',index=False)

In [ ]:
correct = pd.read_csv("correct.csv")
copy = correct.copy()

## Rest from inat

In [ ]:
inat_rest = inat[~inat.isin(inat_obs) & ~(inat.isin(inat_taxa))]

In [ ]:
len(inat_rest)

2

In [ ]:
new_rows = pd.DataFrame({
    "Invalid_link": inat_rest,
    "category": "rest_inat",
    "correct": pd.NA
})


In [ ]:
correct = pd.concat([correct, new_rows], ignore_index=True).to_csv('correct.csv',index=False)

In [ ]:
correct = pd.read_csv("correct.csv")
copy = correct.copy()

## Rest from https

In [ ]:
http_r_df = http_df[~http_df.isin(inat)]
len(http_r_df)

2

In [ ]:
http_r_df

14334    https://inaturalist.lu//observations/338544252
14335       https://inaturalist.lu//observations/338544
Name: field:url for "partner" observation, dtype: str

In [ ]:
new_rows = pd.DataFrame({
    "Invalid_link": http_r_df,
    "category": "rest_http",
    "correct": pd.NA
})


In [ ]:
correct = pd.concat([correct, new_rows], ignore_index=True).to_csv('correct.csv',index=False)

In [ ]:
correct = pd.read_csv("correct.csv")
copy = correct.copy()

## invalid_urls - contains obs

In [ ]:
invalid_urls = invalid_urls.dropna().reset_index(drop=True)

In [ ]:
invalid_rest =invalid_urls[~invalid_urls.isin(http_df)]
len(invalid_rest)


59

In [ ]:
invalid = invalid_rest[invalid_rest.str.lower() != "na"]
len(invalid)

52

In [ ]:
invalid_try = invalid[invalid.str.contains(r"observations")]
invalid_try

12     inaturalist.org/observations/107621367; https:...
63     1https://www.inaturalist.org/observations/2060...
103           www.inaturalist.org/observations/287993063
116               inaturalist.org/observations/347352736
Name: field:url for "partner" observation, dtype: str

In [ ]:
new_rows = pd.DataFrame({
    "Invalid_link": invalid_try,
    "category": "http_obs",
    "correct": pd.NA
})


In [ ]:
correct = pd.concat([correct, new_rows], ignore_index=True).to_csv('correct.csv',index=False)

In [ ]:
correct = pd.read_csv("correct.csv")
copy = correct.copy()

## invalid urls - contains digits

In [ ]:
digits = invalid_rest[invalid_rest.str.contains(r"[0-9]", na=False)]
digits = digits[~(digits.isin(invalid_try))]
digits

10     97610245
11     97610244
83    227722002
84    227602079
Name: field:url for "partner" observation, dtype: str

In [ ]:
new_rows = pd.DataFrame({
    "Invalid_link": digits,
    "category": "digits",
    "correct": pd.NA
})

In [ ]:
correct = pd.concat([correct, new_rows], ignore_index=True).to_csv('correct.csv',index=False)

In [ ]:
correct = pd.read_csv("correct.csv")
copy = correct.copy()

## Rest invalid

In [ ]:
invalid = invalid[~(invalid.isin(digits))]
invalid = invalid[~(invalid.isin(invalid_try))]
len(invalid)

44

In [ ]:
new_rows = pd.DataFrame({
    "Invalid_link": invalid,
    "category": "invalid",
    "correct": pd.NA
})

In [ ]:
correct = pd.concat([correct, new_rows], ignore_index=True).to_csv('correct.csv',index=False)

In [ ]:
correct = pd.read_csv("correct.csv")
copy = correct.copy()

# Convert into valid urls


In [464]:
# correct = pd.read_csv("correct.csv")
correct.head()

,Invalid_link,category,correct
0,https://www.inaturalist.org/observations/43116...,obs,https://www.inaturalist.org/observations/4311650
1,https://www.inaturalist.org/observations/14425...,obs,https://www.inaturalist.org/observations/14425...
2,https://www.inaturalist.org/observations/new?c...,obs,https://www.inaturalist.org/observations/51188925
3,https://www.inaturalist.org/observations/11339...,obs,https://www.inaturalist.org/observations/11339...
4,https://www.inaturalist.org/observations/14778...,obs,https://www.inaturalist.org/observations/14778...


In [465]:
correct =correct[~correct["Invalid_link"].duplicated()]


In [466]:
lookup = correct.set_index("Invalid_link")["correct"]
lookup[:-10]

Invalid_link
https://www.inaturalist.org/observations/4311650#activity_comment_2442449                                                                                                                                          https://www.inaturalist.org/observations/4311650
https://www.inaturalist.org/observations/144250537#activity_comment_6ca468b2-ec39-409e-bc11-3af829ae81fa                                                                                                          https://www.inaturalist.org/observations/14425...
https://www.inaturalist.org/observations/new?copy=51188925                                                                                                                                                        https://www.inaturalist.org/observations/51188925
https://www.inaturalist.org/observations/113398823: https://www.inaturalist.org/observations/113401806; https://www.inaturalist.org/observations/113730771; https://www.inaturalist.org/observations/209436116 

In [467]:
correct[correct["correct"]=="nahi"]


,Invalid_link,category,correct
16,https://www.inaturalist.org/observations?on=20...,obs,nahi
17,https://www.inaturalist.org/taxa/91566-Pterygo...,taxa,nahi
18,https://www.inaturalist.org/taxa/53511-Monopte...,taxa,nahi
19,https://www.inaturalist.org/taxa/85876-Loricar...,taxa,nahi
20,https://www.inaturalist.org/guide_taxa/1201268,taxa,nahi
...,...,...,...
105,ants,invalid,nahi
106,Ophidiaster ophidianus,invalid,nahi
108,Leiaster speciosus,invalid,nahi
109,don't have one bc the tree is cultivated,invalid,nahi


In [468]:
df_temp[col] = df_temp[col].map(lookup).fillna(df_temp[col])
# df_temp[[col]=="https://www.inaturalist.org/observations/4311650"]

In [469]:
df_temp[df_temp["scientific_name"].str.contains("Aphididae",na=False)][col]

997      https://www.inaturalist.org/observations/52273719
1605     https://www.inaturalist.org/observations/11145464
2008     https://www.inaturalist.org/observations/68682858
2153     https://www.inaturalist.org/observations/70720212
2309                                                   NaN
2476     https://www.inaturalist.org/observations/78321781
2479     https://www.inaturalist.org/observations/78337807
2562     https://www.inaturalist.org/observations/80432584
2577                                                   NaN
3316     https://www.inaturalist.org/observations/10495...
3840     https://www.inaturalist.org/observations/11905...
3972     https://www.inaturalist.org/observations/12281...
4336     https://www.inaturalist.org/observations/13628...
4452     https://www.inaturalist.org/observations/13997...
4466     https://www.inaturalist.org/observations/14015...
4489     https://www.inaturalist.org/observations/14078...
4622     https://www.inaturalist.org/observations/14380.

In [470]:
df_temp[df_temp[col].str.contains("Aphididae",na=False)][[col, "id","scientific_name"]]

,"field:url for ""partner"" observation",id,scientific_name


In [471]:
df_temp[col] =df_temp[col].replace("nahi",pd.NA)

## Second check

In [479]:

pattern = (
    r'(?i)^\s*https?://(?:www\.)?'
    r'(?:(?:[a-z0-9-]+\.)?inaturalist\.org'
    r'|inaturalist\.[a-z]+(?:\.[a-z]+){0,2}'
    r'|naturalista\.mx'
    r'|biodiversity4all\.org)'
    r'/observations(?:/\d+)?(?:\?.*)?'
    r'\s*$'
)



# 2) Work with a Series; strip spaces to make matching resilient
urls_try = df_temp[col].astype(str).str.strip()

# 3) Build mask of rows that DO NOT match the required observation URL format
invalid_mask = ~(urls_try.str.fullmatch(pattern))

# 4) Get the invalid URLs as a Series
invalid_urls = urls_try[invalid_mask]

#5) Remove "nan" values
invalid_urls = invalid_urls[invalid_urls.str.lower() != 'nan']

In [480]:
# invalid_urls.isna().sum()
invalid_urls = invalid_urls[~(invalid_urls.isna())]


In [481]:
len(invalid_urls)

195

In [482]:
invalid_urls

23       https://www.inaturalist.org/observations/66517...
54       https://www.inaturalist.org/observations/66517...
209      https://www.inaturalist.org/observations/88362...
240      https://www.inaturalist.org/observations/94595...
321      https://www.inaturalist.org/observations/11243...
                               ...                        
14235    https://www.inaturalist.org/observations/33763...
14625    https://www.inaturalist.org/observations/34206...
14806    https://www.inaturalist.org/observations/34610...
14809    https://www.inaturalist.org/observations/34610...
14883    https://www.inaturalist.org/observations/34766...
Name: field:url for "partner" observation, Length: 195, dtype: str

In [483]:
new_rows = pd.DataFrame({
    "Invalid_link": invalid_urls,
    "category": "concat",
    "correct": pd.NA
})
new_rows.to_csv("delete.csv",index=False)

In [477]:
correct = pd.concat([correct, new_rows], ignore_index=True).to_csv('correct1.csv',index=False)

# Linking obs and partner

In [503]:

# Store the partner_IDs from the partner observation column
pattern = r'(?:observations/)?(\d+)\b'

# Handle missing values by replacing them with an empty string
df_temp['partner_ids'] = df_temp[col].apply(lambda x: re.findall(pattern, str(x) if pd.notna(x) else ''))

df_temp[['partner_ids', col]]

,partner_ids,"field:url for ""partner"" observation"
0,[1766953],https://www.inaturalist.org/observations/1766953
1,[],NaN
2,[414707],https://www.inaturalist.org/observations/414707
3,[414645],https://www.inaturalist.org/observations/414645
4,[],NaN
...,...,...
14894,[347744274],https://www.inaturalist.org/observations/34774...
14895,[347744271],https://www.inaturalist.org/observations/34774...
14896,[347744291],https://www.inaturalist.org/observations/34774...
14897,[347744290],https://www.inaturalist.org/observations/34774...


In [504]:
link_df = df_temp[df_temp[col].notna()]
link_df[['partner_ids', col]]

,partner_ids,"field:url for ""partner"" observation"
0,[1766953],https://www.inaturalist.org/observations/1766953
2,[414707],https://www.inaturalist.org/observations/414707
3,[414645],https://www.inaturalist.org/observations/414645
6,[66466718],https://www.inaturalist.org/observations/66466718
8,[66466454],https://www.inaturalist.org/observations/66466454
...,...,...
14893,[347741350],https://www.inaturalist.org/observations/34774...
14894,[347744274],https://www.inaturalist.org/observations/34774...
14895,[347744271],https://www.inaturalist.org/observations/34774...
14896,[347744291],https://www.inaturalist.org/observations/34774...


In [505]:
len(link_df)

12008

In [506]:
link_df= link_df.explode('partner_ids').reset_index(drop=True)
link_df.shape

(12378, 9)

In [507]:
link_df.to_csv("link1.csv",index=False)

In [502]:
link_df['partner_ids'] = pd.to_numeric(
    link_df['partner_ids'], errors='coerce'
).astype('Int64')

In [492]:
columns=["id","partner_ids","scientific_name","partner_scientific_name","url","partner_url"]
final_df = pd.DataFrame(columns=columns)

In [494]:
final_df.columns

Index(['id', 'partner_ids', 'scientific_name', 'partner_scientific_name',
       'url', 'partner_url'],
      dtype='str')

In [495]:
# Iterate through each row in the expanded DataFrame and build the final DataFrame accordingly
rows = []
for index, row in link_df.iterrows():
    l=[]

    new_row = {
    "id" : row['id'],
    "url" : row['url'],
    "scientific_name": row['scientific_name']
    }

    x = link_df[link_df['id'] == row['partner_ids']]

    if not x.empty:
        x = x.iloc[0]  # take the first match
        new_row['partner_id'] = x['id']
        new_row['partner_scientific_name'] = x['scientific_name']
        new_row['partner_url'] = x['url']

    else:
        # fallback if prey not found
        new_row['partner_id'] = None
        new_row['partner_scientific_name'] = None
        new_row['partner_url'] = None


        # Append row to final dataframe
        #final_df = pd.concat([final_df, pd.DataFrame([new_row])], ignore_index=True)
        rows.append(new_row)
    final_df = pd.DataFrame(rows, columns=columns)

In [497]:
final_df.to_csv('final_df.csv', index=False)